# Bibliotheque + lecture Excel

In [1]:
import pandas as pd 
import numpy as np 
import os
import re
import datetime
from collections import defaultdict
import zipfile
from datetime import datetime, timedelta
from pathlib import Path

In [2]:
df = pd.read_excel('AGING_Ceinture_Montre (avec groupe).xlsx',sheet_name=None)

C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():
C:\Users\judupont\anaconda3\Lib\site-packages\openpyxl\worksheet\_read_only.py:85: UserWarning: Conditional Formatting extension is not supported and will be removed
  for idx, row in parser.parse():


In [3]:
df.keys()

dict_keys(['APHM', 'CAEN', 'POITIERS', 'ROUEN', 'LAVERAN', 'LPC', 'SAINTE-MARGUERITE', 'CGD', 'CERCA', 'Effectifs ceinture', 'Effectifs montre', 'CODEBOOK '])

# Vérification blocs

In [4]:
 # ===== Conversion heures =====
def convertir_heure_excel_ou_texte(x):
     if pd.isna(x):
        return pd.NaT
     if isinstance(x, (int, float)):
        return pd.to_datetime(x, unit="D", origin="1899-12-30")
     try:
        return pd.to_datetime(str(x).strip(), format="%H:%M")
     except Exception:
         try:
             return pd.to_datetime(str(x).strip())
         except Exception:
            return pd.NaT

In [5]:
def analyser_feuille_bloc(
    df,
    nom_feuille,
    col_debut,
    col_fin,
    nom_bloc
):
    print(f"\n--- Feuille : {nom_feuille} | {nom_bloc} ---")

    col_id = "Numero_inclusion"

    # ===== Dictionnaire des règles par centre / bloc / candidat =====
    REGLES_CANDIDATS = {
        ("CAEN", "bloc2", None): {"col_fin": "heure_rappel_cond_v4"},
        ("CAEN", "bloc3", None): {"col_debut": "heure_rappel_cond_v4"},
        ("CAEN", "bloc2", "0326BJR"): {"col_fin": "heure_fluence_sem_v4"},
        ("CAEN", "bloc3", "0326BJR"): {"col_debut": "heure_fluence_sem_v4"},

        ("POITIERS", 'bloc1', "0406MCS"): {"col_debut": "heure_montre_v4"},
        ("POITIERS", 'bloc2', "0408BCS"): {"col_fin": "heure_rappel_cond_v4"},
        ("POITIERS", 'bloc3', "0408BCS"): {"col_debut": "heure_rappel_cond_v4"},

        ("LPC", 'bloc2', "0112DRR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0112DRR"): {"col_debut": "heure_rappel_cond_v4"},
        ("LPC", 'bloc2', "0115MHR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0115MHR"): {"col_debut": "heure_rappel_cond_v4"},
        ("LPC", 'bloc2', "0141HJR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0141HJR"): {"col_debut": "heure_rappel_cond_v4"},
        ("LPC", 'bloc2', "0144ZGR"): {"col_fin": "heure_rappel_cond_v4"},
        ("LPC", 'bloc3', "0144ZGR"): {"col_debut": "heure_rappel_cond_v4"},

        ("CERCA", 'bloc2', "0406BBS"): {"col_fin": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc3', "0406BBS"): {"col_debut": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc2', "0408SJS"): {"col_fin": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc3', "0408SJS"): {"col_debut": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc2', "0409HCR"): {"col_fin": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc3', "0409HCR"): {"col_debut": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc2', "0410FMS"): {"col_fin": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc3', "0410FMS"): {"col_debut": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc2', "0418TJR"): {"col_fin": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc3', "0418TJR"): {"col_debut": "heure_rappel_cond_v4"},
        ("CERCA", 'bloc2', "0420MCR"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0420MCR"): {"col_debut": "heure_nback_v4"},
        ("CERCA", 'bloc2', "0420MCR"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0420MCR"): {"col_debut": "heure_nback_v4"},
        ("CERCA", 'bloc2', "0423SVS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0423SVS"): {"col_debut": "heure_nback_v4"},
        ("CERCA", 'bloc2', "0432CGS"): {"col_fin": "heure_nback_v4"},
        ("CERCA", 'bloc3', "0432CGS"): {"col_debut": "heure_nback_v4"},
        
    }

    # ===== Colonnes effectives par défaut =====
    col_debut_effective = col_debut
    col_fin_effective = col_fin

    # ===== Application des règles =====
    for (centre, bloc, candidat), regle in REGLES_CANDIDATS.items():
        if centre != nom_feuille:
            continue
        if bloc is not None and bloc != nom_bloc:
            continue
        if candidat is not None:
            mask_candidat = df[col_id] == candidat
            if not mask_candidat.any():
                continue

        # Colonnes à remplacer
        if "col_debut" in regle:
            col_debut_effective = regle["col_debut"]
        if "col_fin" in regle:
            col_fin_effective = regle["col_fin"]

        print(f"🔧 Règle appliquée → centre={centre}, bloc={bloc}, candidat={candidat}")

    # ===== Vérification colonnes =====
    for col in [col_debut_effective, col_fin_effective, col_id]:
        if col not in df.columns:
            print(f"❌ Colonne manquante : {col}")
            return None

    # ===== Copie du DataFrame =====
    df = df.copy()

    # Conversion UNIQUEMENT des colonnes utilisées
    df[col_debut_effective] = df[col_debut_effective].apply(convertir_heure_excel_ou_texte)
    df[col_fin_effective] = df[col_fin_effective].apply(convertir_heure_excel_ou_texte)

    # ===== Durée =====
    col_duree = f"duree_{nom_bloc}"
    col_rejet = f"rejeter_{nom_bloc}"

    df[col_duree] = (df[col_fin_effective] - df[col_debut_effective]).dt.total_seconds() / 60

    # Passage de minuit
    df.loc[df[col_duree] < 0, col_duree] += 24 * 60

    # ===== Masques =====
    mask_manquant = df[col_duree].isna()
    mask_valide = df[col_duree].notna() & (df[col_duree] > 0)

    # ===== Statistiques =====
    moyenne = df.loc[mask_valide, col_duree].mean()
    ecart_type = df.loc[mask_valide, col_duree].std()
    print(f"Moyenne = {moyenne:.2f} min")
    print(f"Écart-type = {ecart_type:.2f} min")

    borne_inf = moyenne - 2 * ecart_type

    # ===== Rejets =====
    df[col_rejet] = 0
    df.loc[mask_manquant, col_rejet] = 2

    mask_rejet = df[col_duree] < borne_inf

    # Règle spécifique bloc 1
    if nom_bloc == "bloc1":
        mask_rejet = mask_rejet | (df[col_duree] < 10)

    df.loc[mask_valide & mask_rejet, col_rejet] = 1

    # ===== Résumé =====
    print(f"\nRésumé {col_rejet} :")
    print(df[col_rejet].value_counts().sort_index())

    return {
        "df": df,
        "moyenne": moyenne,
        "ecart_type": ecart_type,
        "rejets": {
            "manquant": df.loc[df[col_rejet] == 2, col_id].tolist(),
            "rejetes": df.loc[df[col_rejet] == 1, col_id].tolist(),
        }
    }


## Bloc 1

In [6]:
feuilles = [
    'APHM', 'CAEN', 'POITIERS', 'ROUEN',
    'LAVERAN', 'LPC', 'SAINTE-MARGUERITE',
    'CGD', 'CERCA'
]

resultats_bloc1 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc1[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_montre_v4",   
        col_fin="heure_anamnese_fin_v4",        
        nom_bloc="bloc1"
    )




--- Feuille : APHM | bloc1 ---
Moyenne = 19.61 min
Écart-type = 9.05 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    14
1     4
2     7
Name: count, dtype: int64


--- Feuille : CAEN | bloc1 ---
Moyenne = 13.50 min
Écart-type = 4.36 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    39
1     9
2     8
Name: count, dtype: int64


--- Feuille : POITIERS | bloc1 ---
🔧 Règle appliquée → centre=POITIERS, bloc=bloc1, candidat=0406MCS
Moyenne = 20.08 min
Écart-type = 6.76 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    12
2     3
Name: count, dtype: int64


--- Feuille : ROUEN | bloc1 ---
Moyenne = 17.53 min
Écart-type = 6.74 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    14
1     1
Name: count, dtype: int64


--- Feuille : LAVERAN | bloc1 ---
Moyenne = 21.75 min
Écart-type = 2.22 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    4
2    1
Name: count, dtype: int64


--- Feuille : LPC | bloc1 ---
Moyenne = 19.44 min
Écart-type = 8.31 min

Résumé rejeter_bloc1 :
rejeter_bloc1
0    41
1     2
2     1
N

## Bloc 2 

In [7]:
resultats_bloc2 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc2[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_rlri16imm_debut_v4",  
        col_fin= "heure_nback_v4 (consignes)" ,         
        nom_bloc="bloc2"
    )




--- Feuille : APHM | bloc2 ---
Moyenne = 46.72 min
Écart-type = 9.14 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    18
2     7
Name: count, dtype: int64


--- Feuille : CAEN | bloc2 ---
🔧 Règle appliquée → centre=CAEN, bloc=bloc2, candidat=None
🔧 Règle appliquée → centre=CAEN, bloc=bloc2, candidat=0326BJR
Moyenne = 44.57 min
Écart-type = 6.41 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    48
1     1
2     7
Name: count, dtype: int64


--- Feuille : POITIERS | bloc2 ---
🔧 Règle appliquée → centre=POITIERS, bloc=bloc2, candidat=0408BCS
Moyenne = 46.00 min
Écart-type = 10.40 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    12
2     3
Name: count, dtype: int64


--- Feuille : ROUEN | bloc2 ---
Moyenne = 40.47 min
Écart-type = 5.58 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    15
Name: count, dtype: int64


--- Feuille : LAVERAN | bloc2 ---
Moyenne = 40.00 min
Écart-type = 13.14 min

Résumé rejeter_bloc2 :
rejeter_bloc2
0    4
2    1
Name: count, dtype: int64


--- Feuille : LPC | bloc2 --

resultats_bloc2

## Bloc 3 

In [8]:
resultats_bloc3 = {}

for feuille in feuilles:
    print("\n" + "=" * 60)

    resultats_bloc3[feuille] = analyser_feuille_bloc(
        df=df[feuille],
        nom_feuille=feuille,
        col_debut="heure_nback_v4 (consignes)",   # à adapter si besoin
        col_fin="heure_fin_tests_v4",         # à adapter si besoin
        nom_bloc="bloc3"
    )



--- Feuille : APHM | bloc3 ---
Moyenne = 46.33 min
Écart-type = 13.57 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    18
2     7
Name: count, dtype: int64


--- Feuille : CAEN | bloc3 ---
🔧 Règle appliquée → centre=CAEN, bloc=bloc3, candidat=None
🔧 Règle appliquée → centre=CAEN, bloc=bloc3, candidat=0326BJR
Moyenne = 61.45 min
Écart-type = 198.00 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    49
2     7
Name: count, dtype: int64


--- Feuille : POITIERS | bloc3 ---
🔧 Règle appliquée → centre=POITIERS, bloc=bloc3, candidat=0408BCS
Moyenne = 43.50 min
Écart-type = 13.26 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    12
2     3
Name: count, dtype: int64


--- Feuille : ROUEN | bloc3 ---
Moyenne = 44.67 min
Écart-type = 6.21 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    14
1     1
Name: count, dtype: int64


--- Feuille : LAVERAN | bloc3 ---
Moyenne = 41.75 min
Écart-type = 9.03 min

Résumé rejeter_bloc3 :
rejeter_bloc3
0    4
2    1
Name: count, dtype: int64


--- Feuille : LPC | bloc3 

# Nombre de candidats qui passent toutes les conditions 

In [9]:
def candidats_valides_tous_blocs_depuis_resultats(
    feuille,
    resultats_blocs,
    col_id="Numero_inclusion"
):
    """
    resultats_blocs = dict {
        "bloc1": resultats_bloc1,
        "bloc2": resultats_bloc2,
        "bloc3": resultats_bloc3
    }
    """

    # Récupération du df de référence (bloc1 par ex)
    df_ref = resultats_blocs["bloc1"][feuille]["df"].copy()

    colonnes_rejet = []

    # Ajouter chaque colonne de rejet depuis chaque bloc
    for nom_bloc, res_bloc in resultats_blocs.items():
        col_rejet = f"rejeter_{nom_bloc}"
        if col_rejet not in res_bloc[feuille]["df"].columns:
            print(f"Colonne manquante : {col_rejet} dans {feuille}")
            return None

        df_ref[col_rejet] = res_bloc[feuille]["df"][col_rejet]
        colonnes_rejet.append(col_rejet)

    # Condition : tout à 0
    mask_valide = (df_ref[colonnes_rejet] == 0).all(axis=1)

    candidats_ok = df_ref.loc[mask_valide, col_id].tolist()
    candidats_rejetes = df_ref.loc[~mask_valide, col_id].tolist()

    print(f"\n=== {feuille} ===")
    print(f"Candidats valides sur TOUS les blocs : {len(candidats_ok)}")

    print("Liste des candidats conservés :")
    for pid in candidats_ok:
        print(f" - {pid}")

    return {
        "nb_valides": len(candidats_ok),
        "valides": candidats_ok,
        "rejetes": candidats_rejetes,
    }


In [10]:
resultats_globaux = {}

colonnes_blocs = [
    "rejeter_bloc1",
    "rejeter_bloc2",
    "rejeter_bloc3"
]

for feuille in feuilles:
    print("\n" + "=" * 70)
    print(f"Analyse globale – Feuille : {feuille}")

    # ===== Base : bloc 1 =====
    df_global = resultats_bloc1[feuille]["df"][
        ["Numero_inclusion", "rejeter_bloc1"]
    ].copy()

    # ===== Fusion bloc 2 =====
    df_global = df_global.merge(
        resultats_bloc2[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc2"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Fusion bloc 3 =====
    df_global = df_global.merge(
        resultats_bloc3[feuille]["df"][
            ["Numero_inclusion", "rejeter_bloc3"]
        ],
        on="Numero_inclusion",
        how="left"
    )

    # ===== Affichage debug AVANT filtre =====
    print("\n📊 df_global AVANT fillna et filtres :")
    display(df_global)

    # ===== Sécurité : NaN → rejet (2) =====
    df_global[colonnes_blocs] = df_global[colonnes_blocs].fillna(2)

    mask = (
        (df_global["rejeter_bloc1"] == 0) &
        (df_global["rejeter_bloc2"] == 0) &
        (df_global["rejeter_bloc3"] == 0)
    )

    # ===== Extraction =====
    candidats_valides = df_global.loc[
        mask, "Numero_inclusion"
    ].tolist()

    candidats_rejetes = df_global.loc[
        ~mask, "Numero_inclusion"
    ].tolist()

    # ===== Résumés =====
    print(f"\nNombre total de candidats : {len(df_global)}")
    print(f"Candidats VALIDES (selon Condition) : {len(candidats_valides)}")
    print(f"Candidats REJETÉS : {len(candidats_rejetes)}")

    print("\nListe des candidats valides :")
    for pid in candidats_valides:
        print(f" - {pid}")

    # ===== Stockage =====
    resultats_globaux[feuille] = {
        "df": df_global,
        "valides": candidats_valides,
        "rejetes": candidats_rejetes,
        "n_valides": len(candidats_valides),
        "n_rejetes": len(candidats_rejetes),
    }



Analyse globale – Feuille : APHM

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0101CAR,0,0,0
1,0102PCR,0,0,0
2,0103SHS,1,0,0
3,0105PNR,0,0,0
4,0104FJS,0,0,0
5,0106JLS,1,0,0
6,0107DSS,0,0,0
7,0108BFS,1,0,0
8,0109GSS,1,0,0
9,0110LPR,0,0,0



Nombre total de candidats : 25
Candidats VALIDES (selon Condition) : 14
Candidats REJETÉS : 11

Liste des candidats valides :
 - 0101CAR
 - 0102PCR
 - 0105PNR
 - 0104FJS
 - 0107DSS
 - 0110LPR
 - 0111MNR
 - 0112BSR
 - 0114LMS
 - 0116VAR
 - 0119LJS
 - 0121MJR
 - 0122BDS
 - 0123RMS

Analyse globale – Feuille : CAEN

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0301GNR,2,2,2
1,0302ZMR,0,0,0
2,0303PAR,2,2,2
3,0304VMR,0,0,0
4,0305LCR,0,0,0
5,0307SMR,0,0,0
6,0306MDR,1,0,0
7,0308RGR,0,0,0
8,0309DBS,2,2,2
9,0311CJS,0,0,0



Nombre total de candidats : 56
Candidats VALIDES (selon Condition) : 37
Candidats REJETÉS : 19

Liste des candidats valides :
 - 0302ZMR
 - 0304VMR
 - 0305LCR
 - 0307SMR
 - 0308RGR
 - 0311CJS
 - 0310ACS
 - 0315VCS
 - 0313FPR
 - 0316TMS
 - 0317LGS
 - 0319LJS
 - 0321DRR
 - 0323RFS
 - 0322RMS
 - 0324LMR
 - 0328MPR
 - 0330RTR
 - 0329NMR
 - 0331GRS
 - 0333MMS
 - 0334TVS
 - 0337BFS
 - 0338JBS
 - 0339NPR
 - 0340BAR
 - 0341LJS
 - 0342VNS
 - 0343PAS
 - 0345LAR
 - 0347DMR
 - 0346GLS
 - 0348GCR
 - 0350MYR
 - 0351FIR
 - 0352RNR
 - 0356DMS

Analyse globale – Feuille : POITIERS

📊 df_global AVANT fillna et filtres :


...


Nombre total de candidats : 8
Candidats VALIDES (selon Condition) : 7
Candidats REJETÉS : 1

Liste des candidats valides :
 - 0801HDR
 - 0802LAS
 - 0804GOR
 - 0803DPS
 - 0805BMS
 - 0806KHS
 - 0807OMR

Analyse globale – Feuille : CGD

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0901SMR,0,0,0
1,0903DJS,0,0,0
2,0902SIS,0,0,0
3,0906DNR,2,2,2
4,0904KJS,0,0,0
5,0905SLR,0,0,0



Nombre total de candidats : 6
Candidats VALIDES (selon Condition) : 5
Candidats REJETÉS : 1

Liste des candidats valides :
 - 0901SMR
 - 0903DJS
 - 0902SIS
 - 0904KJS
 - 0905SLR

Analyse globale – Feuille : CERCA

📊 df_global AVANT fillna et filtres :


,Numero_inclusion,rejeter_bloc1,rejeter_bloc2,rejeter_bloc3
0,0401BDS,0,0,0
1,0402TFR,2,2,2
2,0404DER,0,0,0
3,0405RCR,0,0,0
4,0406BBS,0,0,0
5,0407HMS,2,2,2
6,0408SJS,0,0,0
7,0409HCR,0,0,0
8,0410FMS,0,0,0
9,0411NPR,2,2,2



Nombre total de candidats : 40
Candidats VALIDES (selon Condition) : 12
Candidats REJETÉS : 28

Liste des candidats valides :
 - 0401BDS
 - 0404DER
 - 0405RCR
 - 0406BBS
 - 0408SJS
 - 0409HCR
 - 0410FMS
 - 0413MMR
 - 0414PVR
 - 0418MPR
 - 0420MCR
 - 0423SVS


In [11]:
total_valides = sum(
    res["n_valides"] for res in resultats_globaux.values()
)

print("=" * 70)
print(f"✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : {total_valides}")

✅ Nombre TOTAL de candidats valides (toutes feuilles confondues) : 143


# 143 candidats retenus

# Recherche des candidats et téléchargement de leur fichier EDA V4

In [12]:
import pandas as pd

# Afficher toutes les lignes
pd.set_option('display.max_rows', None)

# Afficher toutes les colonnes
pd.set_option('display.max_columns', None)

# Afficher toute la largeur de chaque colonne
pd.set_option('display.max_colwidth', None)

## APHM

In [13]:
# ===== Racine des patients =====
racine_aphm = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"

# ===== Candidats valides =====
candidats_aphm = [c.upper() for c in resultats_globaux["APHM"]["valides"]]

rows = []

for root, dirs, files in os.walk(racine_aphm):

    if not any("V4" in d.upper() for d in root.split(os.sep)):
        continue

    for f in files:
        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper(): 
            continue

        chemin = os.path.join(root, f)

        # ===== Numero_inclusion = dossier juste après la racine =====
        rel_path = os.path.relpath(chemin, racine_aphm)
        parts = rel_path.split(os.sep)
        pid_trouve = parts[0].upper()

        if pid_trouve not in candidats_aphm:
            continue

        # ===== Timestamp Unix (dossier juste avant EDA.csv) =====
        folder = os.path.basename(os.path.dirname(chemin))
        timestamp_match = re.match(r"(\d+)_", folder)
        timestamp_unix = int(timestamp_match.group(1)) if timestamp_match else None

        # ===== Conversion Unix → date locale =====
        date_eda = datetime.datetime.fromtimestamp(timestamp_unix) if timestamp_unix else None

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Feuille": "APHM",
            "Timestamp_unix": timestamp_unix,
            "Date_eda": date_eda
        })

# ===== DataFrame =====
df_aphm = pd.DataFrame(rows, columns=["Numero_inclusion", "Chemin", "Feuille", "Timestamp_unix", "Date_eda"])

# ===== Suppression des doublons =====
if not df_aphm.empty:
    df_aphm_final = (
        df_aphm
        .assign(priorite_montre=df_aphm["Chemin"].str.contains("MONTRE", case=False))
        .sort_values(["Numero_inclusion", "priorite_montre"], ascending=[True, False])
        .drop_duplicates(subset="Numero_inclusion", keep="first")
        .drop(columns="priorite_montre")
        .reset_index(drop=True)
    )
else:
    df_aphm_final = df_aphm.copy()

print(f"Candidats attendus APHM : {len(candidats_aphm)}")
print(f"Fichiers EDA APHM V2 retenus : {len(df_aphm_final)}")

display(df_aphm_final)


Candidats attendus APHM : 14
Fichiers EDA APHM V2 retenus : 14


,Numero_inclusion,Chemin,Feuille,Timestamp_unix,Date_eda
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 04-04-2019 V4\Montre 0101CAR\1554360113_A01093\EDA.csv,APHM,1.554360e+09,2019-04-04 08:41:53
1,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 12-04-2019 V4\Montre 0102PCR\1555052819_A01093\EDA.csv,APHM,1.555053e+09,2019-04-12 09:06:59
2,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 08-11-2019 V4\Montre 0104FJS\1573198120_A01093\EDA.csv,APHM,1.573198e+09,2019-11-08 08:28:40
3,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 09-09-2019 V4\Montre 0105PNR\1568012045_A01093\EDA.csv,APHM,1.568012e+09,2019-09-09 08:54:05
4,0107DSS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0107DSS\0107DSS 17-10-2019 V4\Montre 0107DSS\1571296181_A01093\EDA.csv,APHM,1.571296e+09,2019-10-17 09:09:41
5,0110LPR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0110LPR\0110LPR 21-11-2019 V4\Montre 0110LPR\1574323503_A01093\EDA.csv,APHM,1.574324e+09,2019-11-21 09:05:03
6,0111MNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0111MNR\0111MNR 22-11-2019 V4\Montre 0111MNR\1574410699_A01093\EDA.csv,APHM,1.574411e+09,2019-11-22 09:18:19
7,0112BSR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0112BSR\0112BSR - V4\V4\montre\1583397568_A01093\EDA.csv,APHM,1.583398e+09,2020-03-05 09:39:28
8,0114LMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0114LMS\0114LMS_V4\montre\EDA.csv,APHM,NaN,NaT
9,0116VAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0116VAR\0116VAR_v4\montre\1591255686_A01093\EDA.csv,APHM,1.591256e+09,2020-06-04 09:28:06


## CAEN

In [14]:
# ===== Racine CAEN =====
racine_caen = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN"

# ===== Liste candidats valides =====
candidats_caen = [
    c.upper().replace(" ", "")
    for c in resultats_globaux["CAEN"]["valides"]
]

# =========================================================
# PATHS FORCÉS pour 3 candidats 
# =========================================================

paths_forces = {
    "0346GLS":r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1666942984_A012950346GLS\EDA.csv",
    "0345LAR" : r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1667550736_A012950345LAR\EDA.csv",
    "0322RMS": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1595319263_A012950322RMS\EDA.csv",
    # 0308RGR mal nommé 0308GRR dans l'arborescence
    "0308RGR": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1599030387_A01295 0308GRR\EDA.csv",

    # Inversion dossiers 0342 / 0343
    "0342VNS": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1646730228_A01295 0343PAS\EDA.csv",
    "0343PAS": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1648798691_A01295 0342VNS\EDA.csv",
}

# ===== DataFrame Excel V4 =====
df_excel_caen = df["CAEN"][["Numero_inclusion", "Date_v4"]].copy()
df_excel_caen["Numero_inclusion"] = (
    df_excel_caen["Numero_inclusion"]
    .str.upper()
    .str.strip()
)

df_excel_caen["Date_v4"] = pd.to_datetime(
    df_excel_caen["Date_v4"],
    dayfirst=True,
    errors="coerce"
).dt.date

# ===== Dictionnaire Numero_inclusion -> date =====
dict_date_v4 = dict(zip(
    df_excel_caen["Numero_inclusion"],
    df_excel_caen["Date_v4"]
))

rows = []

for pid, chemin in paths_forces.items():

    if not os.path.exists(chemin):
        print(f"⚠️ Path forcé introuvable : {chemin}")
        continue

    folder = os.path.basename(os.path.dirname(chemin))
    match = re.match(r"(\d+)_A\d+", folder)

    timestamp_unix = int(match.group(1)) if match else None

    date_eda = (
        datetime.datetime.fromtimestamp(timestamp_unix).date()
        if timestamp_unix else None
    )

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin,
        "Feuille" : "CAEN",
        "Timestamp_unix": timestamp_unix,
        "Date_eda": date_eda,
        "Date_v4_excel": dict_date_v4.get(pid)
    })

print("✔ Fichiers forcés ajoutés")

# =========================================================
# Regex Numero_inclusion
# =========================================================

patterns_candidats = {
    pid: re.compile(rf"\b{pid}\b", re.IGNORECASE)
    for pid in candidats_caen
    if pid not in paths_forces
}


for root, dirs, files in os.walk(racine_caen):

    for f in files:

        if not f.lower().endswith(".csv"):
            continue

        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)

        pid_trouve = None
        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                pid_trouve = pid
                break

        if not pid_trouve:
            continue

        folder = os.path.basename(os.path.dirname(chemin))
        match = re.match(r"(\d+)_A\d+", folder)

        timestamp_unix = int(match.group(1)) if match else None

        date_eda = (
            datetime.datetime.fromtimestamp(timestamp_unix).date()
            if timestamp_unix else None
        )

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Feuille" : "CAEN",
            "Timestamp_unix": timestamp_unix,
            "Date_eda": date_eda,
            "Date_v4_excel": dict_date_v4.get(pid_trouve)
        })

# =========================================================
# DATAFRAME
# =========================================================

df_caen = pd.DataFrame(rows)

# =========================================================
# SUPPRESSION DES DOUBLONS
# =========================================================

if not df_caen.empty:

    def choisir_meilleur_fichier(groupe):

        date_excel = groupe["Date_v4_excel"].iloc[0]

        if pd.isna(date_excel):
            return groupe.iloc[0]

        groupe = groupe.copy()

        groupe["ecart_jours"] = (
            pd.to_datetime(groupe["Date_eda"]) -
            pd.to_datetime(date_excel)
        ).abs().dt.days

        exact = groupe[groupe["Date_eda"] == date_excel]
        if not exact.empty:
            return exact.iloc[0]

        return groupe.sort_values("ecart_jours").iloc[0]

    df_caen_final = (
        df_caen
        .groupby("Numero_inclusion", as_index=False)
        .apply(choisir_meilleur_fichier)
        .reset_index(drop=True)
    )

else:
    df_caen_final = df_caen.copy()

# =========================================================
# FILTRE SUR LA DATE
# =========================================================

# convertion en datetime.date
df_caen_final["Date_eda"] = pd.to_datetime(df_caen_final["Date_eda"]).dt.date
df_caen_final["Date_v4_excel"] = pd.to_datetime(df_caen_final["Date_v4_excel"]).dt.date

df_caen_final = df_caen_final[
    df_caen_final["Date_eda"] == df_caen_final["Date_v4_excel"]
].copy()

# =========================================================
# AFFICHAGE
# =========================================================

print(f"\nCandidats attendus CAEN : {len(candidats_caen)}")
print(f"Fichiers EDA CAEN V4 retenus après filtrage exact : {len(df_caen_final)}")

display(df_caen_final)

candidats_trouves = set(df_caen_final["Numero_inclusion"])
candidats_manquants = sorted(set(candidats_caen) - candidats_trouves)

if candidats_manquants:
    print("\n⚠️ Candidats manquants après filtrage :")
    for c in candidats_manquants:
        print(" -", c)
else:
    print("\n✅ Tous les candidats ont été trouvés après filtrage")


✔ Fichiers forcés ajoutés

Candidats attendus CAEN : 37
Fichiers EDA CAEN V4 retenus après filtrage exact : 36


C:\Users\judupont\AppData\Local\Temp\ipykernel_22300\525280060.py:40: UserWarning: Could not infer format, so each element will be parsed individually, falling back to `dateutil`. To ensure parsing is consistent and as-expected, please specify a format.
  df_excel_caen["Date_v4"] = pd.to_datetime(
C:\Users\judupont\AppData\Local\Temp\ipykernel_22300\525280060.py:174: FutureWarning: DataFrameGroupBy.apply operated on the grouping columns. This behavior is deprecated, and in a future version of pandas the grouping columns will be excluded from the operation. Either pass `include_groups=False` to exclude the groupings or explicitly select the grouping columns after groupby to silence this warning.
  .apply(choisir_meilleur_fichier)


,Numero_inclusion,Chemin,Feuille,Timestamp_unix,Date_eda,Date_v4_excel,ecart_jours
0,0302ZMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1571731715_A01295 0302ZMR\EDA.csv,CAEN,1571731715,2019-10-22,2019-10-22,0
1,0304VMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1579856234_A01295 0304VMR\EDA.csv,CAEN,1579856234,2020-01-24,2020-01-24,0
2,0305LCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1579769470_A01295 0305LCR\EDA.csv,CAEN,1579769470,2020-01-23,2020-01-23,0
3,0307SMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1590395693_A01295 0307SMR\EDA.csv,CAEN,1590395693,2020-05-25,2020-05-25,0
4,0308RGR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1599030387_A01295 0308GRR\EDA.csv,CAEN,1599030387,2020-09-02,2020-09-02,0
5,0310ACS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1589787562_A01295 0310ACS\EDA.csv,CAEN,1589787562,2020-05-18,2020-05-18,0
6,0311CJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1592377927_A01295 0311CJS\EDA.csv,CAEN,1592377927,2020-06-17,2020-06-17,0
7,0313FPR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1591603371_A01295 0313FPR\EDA.csv,CAEN,1591603371,2020-06-08,2020-06-08,0
8,0315VCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1591861453_A01295 0315VCS\EDA.csv,CAEN,1591861453,2020-06-11,2020-06-11,0
9,0316TMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-CAEN\ALLAIN\ALLAIN\MONTRE\1591084669_A01295 0316TMS\EDA.csv,CAEN,1591084669,2020-06-02,2020-06-02,0



⚠️ Candidats manquants après filtrage :
 - 0328MPR


## POITIERS

In [16]:
# ===== Racine POITIERS =====
racine_poitiers = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY"

# ===== Candidats valides =====
candidats_poitiers = [
    c.upper().replace(" ", "")
    for c in resultats_globaux["POITIERS"]["valides"]
]

# ===== Regex POITIERS =====
# ex : 0401TSS → match "0401 TS"
patterns_candidats = {
    pid: re.compile(
        rf"{pid[:4]}\s*-?\s*{pid[4:6]}",
        re.IGNORECASE
    )
    for pid in candidats_poitiers
}

fichiers_par_candidat = defaultdict(list)

for root, dirs, files in os.walk(racine_poitiers):

    if "V4" not in root.upper():
        continue

    for f in files:
        if f.upper() != "EDA.CSV":
            continue

        chemin = os.path.join(root, f)

        for pid, pat in patterns_candidats.items():
            if pat.search(chemin):
                fichiers_par_candidat[pid].append(chemin)
                break

rows = []

for pid, fichiers in fichiers_par_candidat.items():

    fichiers = sorted(set(fichiers))  # sécurité doublons exacts

    # priorité MONTRE (logique équivalente à CEINTURE)
    montre = [f for f in fichiers if "montre" in f.lower()]
    chemin_final = montre[0] if montre else fichiers[0]

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Feuille": "POITIERS"
    })

df_poitiers_final = pd.DataFrame(rows)

# ===== Affichage =====
print(f"Candidats attendus POITIERS : {len(candidats_poitiers)}")
print(f"Fichiers EDA POITIERS V2 retenus : {len(df_poitiers_final)}")

manquants = sorted(
    set(candidats_poitiers) - set(df_poitiers_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats POITIERS sont présents")

display(df_poitiers_final)

Candidats attendus POITIERS : 12
Fichiers EDA POITIERS V2 retenus : 10
⚠️ Candidats manquants : ['0414PJS', '0415VMR']


,Numero_inclusion,Chemin,Feuille
0,0401TSS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0401 TS\0401TSS - V4\0401TSS - montre V4\1578646774_A012C6\EDA.csv,POITIERS
1,0402LLS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0402 LL\0402LLS-V4\0402LLS -montre V4\EDA.csv,POITIERS
2,0403DCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0403 DC\0403DCR-V4\0403DCR - montre V4\EDA.csv,POITIERS
3,0404CYS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0404 CY\0404YCS-V4\0404YC-montre V4\EDA.csv,POITIERS
4,0406MCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0406 MC\0406MCS-V4\0406MC -montre V4\EDA.csv,POITIERS
5,0407LJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0407 LJ\0407LJR-V4\0407LJ-montre V4\EDA.csv,POITIERS
6,0408BCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0408 BC\0408 BC-V4\0408BC-montre V4\1596702371_A012C6\EDA.csv,POITIERS
7,0411VES,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0411 VE\0411 VES-V4\0411VE-montre V4\EDA.csv,POITIERS
8,0412ANS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0412 AN\0412 ANS-V4\0412ANS-V4\EDA.csv,POITIERS
9,0413HFS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-POITIERS\MARTIN DE MONTAUDRY\MARTIN DE MONTAUDRY\0413 HF\0413 HFS V4\0413HFS montre V4\EDA.csv,POITIERS


## ROUEN

In [18]:
# ===== Racine ROUEN =====
racine_rouen = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER"

# ===== Candidats valides =====
candidats_rouen = [c.upper() for c in resultats_globaux["ROUEN"]["valides"]]


# ===== Patterns candidats : 05-09, 05-11, etc =====
patterns_candidats = {
    pid: re.compile(rf"\b{pid[:2]}-{pid[2:4]}\b", re.IGNORECASE)
    for pid in candidats_rouen
}

rows = []

# ===== Parcours fichiers =====
for root, dirs, files in os.walk(racine_rouen):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue

        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        pid_trouve = None
        for pid, pat in patterns_candidats.items():
            if pat.search(chemin_upper):
                pid_trouve = pid
                break

        if not pid_trouve:
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Feuille": "ROUEN"
        })

# ===== DataFrame =====
df_rouen_final = pd.DataFrame(rows)

# ===== Suppression des doublons =====
df_rouen_final = (
    df_rouen_final
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# ===== Affichage =====
print(f"Candidats attendus ROUEN : {len(candidats_rouen)}")
print(f"Fichiers EDA ROUEN V2 retenus : {len(df_rouen_final)}")

manquants = sorted(set(candidats_rouen) - set(df_rouen_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats ROUEN manquants :", manquants)
else:
    print("✅ Tous les candidats ROUEN sont présents")

display(df_rouen_final)


Candidats attendus ROUEN : 13
Fichiers EDA ROUEN V2 retenus : 12
⚠️ Candidats ROUEN manquants : ['0509LGS']


,Numero_inclusion,Chemin,Feuille
0,0501MBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-01-MB-S\V4\MONTRE\1574064476_A012E9\EDA.csv,ROUEN
1,0502LIR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-02-LI-R\V4\MONTRE\EDA.csv,ROUEN
2,0503LCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-03-LC-R\V4\MONTRE\1579767764_A012E9\EDA.csv,ROUEN
3,0504BCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-04-BC-S\V4\MONTRE V4\EDA.csv,ROUEN
4,0505PPR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-05-PP-R\V4\MONTRE\EDA.csv,ROUEN
5,0507BDS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-07-BD-S\V4\MONTRE\EDA.csv,ROUEN
6,0508SRR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-08-SR-R\V4\MONTRE\1589267765_A012E9 (2)\EDA.csv,ROUEN
7,0510DFS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-10-DF-S\V4\MONTRE\EDA.csv,ROUEN
8,0512RCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-12-RC-R\V4\MONTRE\EDA.csv,ROUEN
9,0513EBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CHU-ROUEN\HANNIER\HANNIER\05-13-EB-S\V4\MONTRE\1647332236_A012E9\EDA.csv,ROUEN


## LAVERAN

In [20]:
# ===== Racine LAVERAN =====
racine_laveran = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran"
)

# ===== Candidats attendus =====
candidats_laveran = [c.upper() for c in resultats_globaux["LAVERAN"]["valides"]]

rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_laveran):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)

        # ===== Numero_inclusion = 1er dossier après la racine =====
        rel_path = os.path.relpath(chemin, racine_laveran)
        pid = rel_path.split(os.sep)[0].upper()

        if pid not in candidats_laveran:
            continue

        rows.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Feuille": "LAVERAN"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_laveran = pd.DataFrame(rows)

df_laveran_final = (
    df_laveran
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus LAVERAN : {len(candidats_laveran)}")
print(f"Fichiers EDA LAVERAN retenus : {len(df_laveran_final)}")

manquants = sorted(
    set(candidats_laveran) - set(df_laveran_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LAVERAN manquants :", manquants)
else:
    print("✅ Tous les candidats LAVERAN sont présents")

display(df_laveran_final)


OK [0626MCS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V4\0626MCS_v4\MONTRE\1603959501_A01093\EDA.csv
OK [0629TCR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\0629TCR_V4\MONTRE\1605603058_A01093\EDA.csv
OK [0630RJR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V4\MONTRE\1605517689_A01093\EDA.csv
OK [0631LHR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\0631LHR_V4\MONTRE\1605688444_A01093\EDA.csv

Candidats attendus LAVERAN : 4
Fichiers EDA LAVERAN retenus : 4
✅ Tous les candidats LAVERAN sont présents


,Numero_inclusion,Chemin,Feuille
0,0626MCS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0626MCS\V4\0626MCS_v4\MONTRE\1603959501_A01093\EDA.csv,LAVERAN
1,0629TCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0629TCR\0629TCR_V4\MONTRE\1605603058_A01093\EDA.csv,LAVERAN
2,0630RJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0630RJR\0630RJR_V4\MONTRE\1605517689_A01093\EDA.csv,LAVERAN
3,0631LHR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0631LHR\0631LHR_V4\MONTRE\1605688444_A01093\EDA.csv,LAVERAN


## LPC

In [21]:
# ===== Racine LPC =====
racine_lpc = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie"
    r"\LNSC\DESHAYES\DESHAYES\DATA_Aging"
)

# ===== Candidats attendus =====
candidats_lpc = [c.upper() for c in resultats_globaux["LPC"]["valides"]]

# ===== CORRECTIONS MANUELLES ARBORESCENCE → EXCEL =====
CORRESPONDANCE_IDS = {
    "0117MSR": "0117MMR"  
}
CORRESPONDANCE_INVERSE = {v: k for k, v in CORRESPONDANCE_IDS.items()}

# ===== PATHS FORCÉS =====
paths_forces = {
    "0130FAR": r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\Marseille_LABO_08decembre2020\DATA_Aging_08decembre2020\0154FAR\V4\0130FAR\montre\1581410797_A01115.zip"
}

rows = []

# =========================================================
# AJOUT DES FICHIERS FORCÉS 
# =========================================================
for pid, chemin in paths_forces.items():

    if not os.path.exists(chemin):
        print(f"⚠️ Path forcé introuvable : {chemin}")
        continue

    chemin_final = chemin  # par défaut

    # 🔥 Si c'est un ZIP → on extrait le premier EDA.csv
    if chemin.lower().endswith(".zip"):
        try:
            with zipfile.ZipFile(chemin, 'r') as zf:

                eda_files = [
                    name for name in zf.namelist()
                    if "EDA" in name.upper() and name.lower().endswith(".csv")
                ]

                if not eda_files:
                    print(f"❌ Aucun EDA trouvé dans {chemin}")
                    continue

                eda_name = eda_files[0]

                dossier_extraction = os.path.dirname(chemin)
                zf.extract(eda_name, dossier_extraction)

                chemin_final = os.path.join(dossier_extraction, eda_name)

                print(f"✅ ZIP extrait pour {pid} → {chemin_final}")

        except zipfile.BadZipFile:
            print(f"❌ ZIP corrompu : {chemin}")
            continue

    rows.append({
        "Numero_inclusion": pid,
        "Chemin": chemin_final,
        "Feuille": "LPC"
    })

print("✔ Fichiers forcés ajoutés")

# =========================================================
# BOUCLE PRINCIPALE 
# =========================================================
for root, dirs, files in os.walk(racine_lpc):

    if "V4" not in root.upper():
        continue

    for f in files:

        # ===== EDA uniquement =====
        if not f.lower().endswith(".csv") and not f.lower().endswith(".zip"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)

        rel_path = os.path.relpath(chemin, racine_lpc)
        pid_arbo = rel_path.split(os.sep)[0].upper()
        pid_excel = CORRESPONDANCE_INVERSE.get(pid_arbo, pid_arbo)

        if pid_excel not in candidats_lpc:
            continue

        if pid_excel in paths_forces:
            continue

        rows.append({
            "Numero_inclusion": pid_excel,
            "Chemin": chemin,
            "Feuille": "LPC"
        })

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_lpc_final = (
    pd.DataFrame(rows)
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus LPC : {len(candidats_lpc)}")
print(f"Fichiers EDA LPC retenus : {len(df_lpc_final)}")

manquants = sorted(set(candidats_lpc) - set(df_lpc_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC sont présents")

display(df_lpc_final)

✅ ZIP extrait pour 0130FAR → C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\Marseille_LABO_08decembre2020\DATA_Aging_08decembre2020\0154FAR\V4\0130FAR\montre\EDA.csv
✔ Fichiers forcés ajoutés

Candidats attendus LPC : 39
Fichiers EDA LPC retenus : 27
⚠️ Candidats LPC manquants : ['0106DJR', '0132GAR', '0133BPS', '0136CJR', '0137BMS', '0138NWR', '0139BMR', '0140LMR', '0141HJR', '0142LLS', '0143EBR', '0144ZGR']


,Numero_inclusion,Chemin,Feuille
0,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\0101EMS V4\Montre\1571730172_A01115\EDA.csv,LPC
1,0102EMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0102EMS\V4 0102EMS\Montre\1567668999_A01115\EDA.csv,LPC
2,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V4 0103BPS\Montre\1568706200_A01115\EDA.csv,LPC
3,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V4 0104IBS\Montre\EDA.csv,LPC
4,0107LER,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0107LER\V4 0107LER\Montre\1567584597_A01115\EDA.csv,LPC
5,0108MMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0108MMS\V4\Montre\1571900786_A01115\EDA.csv,LPC
6,0109MJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0109MJR\V4\Montre\1568360086_A01115\EDA.csv,LPC
7,0110RMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0110RMS\V4\Montre\1568619102_A01115\EDA.csv,LPC
8,0111TAS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0111TAS\V4\Montre\1570087686_A01115\EDA.csv,LPC
9,0112DRR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0112DRR\V4\Montre\1573029333_A01115\EDA.csv,LPC


In [22]:
# ===== Racines =====
racines = [
    r"C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire",
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo"
]

# ===== Candidats LPC partie 2 =====
candidats_lpc2 = [
    c.upper() for c in resultats_globaux["LPC"]["valides"][28:]
]

rows = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for racine in racines:

    print(f"\n--- Scan de : {racine}")

    for root, dirs, files in os.walk(racine):

        root_upper = root.upper()

        if not re.search(r'(^|[^A-Z0-9])V4([^A-Z0-9]|$)', root_upper):
            continue

        if "MONTRE" not in root_upper:
            continue

        for f in files:

            if f.upper() != "EDA.CSV":
                continue
            chemin = os.path.join(root, f)
            chemin_upper = chemin.upper()
            pid_trouve = None
            for c in candidats_lpc2:
                if c in chemin_upper:
                    pid_trouve = c
                    break

            if not pid_trouve:
                print("❌ PID NON TROUVÉ :", chemin)
                continue

            # =================================================
            # DEBUG TEMPS : UNIX (dossier) vs DATE fichier
            # =================================================
            dossier_parent = os.path.basename(os.path.dirname(chemin))
            ts_unix = None
            date_unix_str = "None"

            if "_" in dossier_parent:
                try:
                    ts_unix = int(dossier_parent.split("_")[0])
                    date_unix_str = datetime.datetime.fromtimestamp(
                        ts_unix
                    ).strftime("%Y-%m-%d %H:%M:%S")
                except Exception:
                    pass

            date_fichier_str = datetime.datetime.fromtimestamp(
                os.path.getmtime(chemin)
            ).strftime("%Y-%m-%d %H:%M:%S")

            rows.append({
                "Numero_inclusion": pid_trouve,
                "Chemin": chemin,
                "Feuille": "LPC",
                "Timestamp_unix": ts_unix,
                "Date_unix_path": date_unix_str,
                "Date_modif_csv": date_fichier_str
            })

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
print("\nDEBUG rows length :", len(rows))

if rows:
    df_lpc2_final = (
        pd.DataFrame(rows)
        .sort_values("Chemin")
        .drop_duplicates(subset="Numero_inclusion", keep="first")
        .reset_index(drop=True)
    )
else:
    df_lpc2_final = pd.DataFrame(
        columns=["Numero_inclusion", "Chemin", "Feuille", "Timestamp_unix", "Date_unix_path", "Date_modif_csv"]
    )

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus LPC (partie 2) : {len(candidats_lpc2)}")
print(f"Fichiers EDA retenus              : {len(df_lpc2_final)}")

manquants = sorted(
    set(candidats_lpc2) - set(df_lpc2_final["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats LPC manquants :", manquants)
else:
    print("✅ Tous les candidats LPC partie 2 sont présents")

display(df_lpc2_final)


--- Scan de : C:\Users\judupont\Desktop\data_aging_11_2025_copie\LNSC\LNSC\DESHAYES\DATA_Ancillaire

--- Scan de : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0101EMS\V4\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0101EMS\V4\Montre\1571730172_A01115\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0102BJS\V4 0102BJS\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0103EMS\V4\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 

,Numero_inclusion,Chemin,Feuille,Timestamp_unix,Date_unix_path,Date_modif_csv
0,0132GAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0132GAR\0132GAR_V4\montre\1612774878_A01093\EDA.csv,LPC,1612774878,2021-02-08 10:01:18,2026-02-19 10:52:52
1,0133BPS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0133BPS-0157BPS\0157BPS_V4\montre\1612860803_A01115\EDA.csv,LPC,1612860803,2021-02-09 09:53:23,2026-02-19 10:52:52
2,0136CJR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0136CJR\0136CJR_V4\Montre\1639127858_A01115\EDA.csv,LPC,1639127858,2021-12-10 10:17:38,2026-02-19 10:52:53
3,0138NWR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0138NWR\0138NWR-V4\montre\1649059272_A01115\EDA.csv,LPC,1649059272,2022-04-04 10:01:12,2026-02-19 10:52:53
4,0144ZGR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0144ZGR\0144ZGR_V4\montre\1681200504_A01115\EDA.csv,LPC,1681200504,2023-04-11 10:08:24,2026-02-19 10:52:54
5,0140LMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0168LMR 0140LMR\V4 0140LMR\montre\1676451829_A01115\EDA.csv,LPC,1676451829,2023-02-15 10:03:49,2026-02-19 10:52:56
6,0143EBR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PARTICIPANTS\participants_partie 2_labo\0171EBR _ 0143EBR\0143EBR_V4\montre\1682409900_A01115\EDA.csv,LPC,1682409900,2023-04-25 10:05:00,2026-02-19 10:52:57


## SAINTE-MARGUERITE

In [24]:
# ===== Racine Sainte-Marguerite =====
racine_sm = r"C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite"

# ===== Candidats attendus =====
candidats_sm = [c.upper() for c in resultats_globaux["SAINTE-MARGUERITE"]["valides"]]

# ===== Liste pour stocker les fichiers EDA =====
rows = []

# =========================================================
# BOUCLE SUR LES ZIP
# =========================================================
for root, dirs, files in os.walk(racine_sm):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".zip"):
            continue

        chemin_zip = os.path.join(root, f)

        try:
            pid_trouve = root.split(os.sep)[-3].upper()
        except IndexError:
            print(f"❌ Structure inattendue : {chemin_zip}")
            continue

        if pid_trouve not in candidats_sm:
            print(f"❌ PID NON ATTENDU : {pid_trouve}")
            continue

        # =====================================================
        # Dézipper dans le dossier du ZIP lui-même
        # =====================================================
        try:
            with zipfile.ZipFile(chemin_zip, 'r') as zf:

                eda_files = [
                    name for name in zf.namelist()
                    if "EDA" in name.upper() and name.lower().endswith(".csv")
                ]

                if not eda_files:
                    print(f"❌ Pas de fichier EDA dans : {chemin_zip}")
                    continue

                for eda_name in eda_files:

                    zf.extract(eda_name, root)
                    eda_path = os.path.join(root, eda_name)

                    rows.append({
                        "Numero_inclusion": pid_trouve,
                        "Chemin": eda_path,
                        "Feuille": "SAINTE-MARGUERITE"
                    })

                    print(f"✅ EDA extrait [{pid_trouve}] :", eda_path)

        except zipfile.BadZipFile:
            print(f"❌ ZIP corrompu : {chemin_zip}")

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_sm = pd.DataFrame(rows)

df_sm = (
    df_sm
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

print(f"\nNombre de fichiers EDA extraits Sainte-Marguerite : {len(df_sm)}")
display(df_sm)

✅ EDA extrait [0801HDR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V4\Montre\EDA.csv
✅ EDA extrait [0802LAS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V4\Montre\EDA.csv
✅ EDA extrait [0803DPS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS-V4\Montre\EDA.csv
✅ EDA extrait [0804GOR] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR_V4\Montre\EDA.csv
✅ EDA extrait [0805BMS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V4\Montre\EDA.csv
✅ EDA extrait [0806KHS] : C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0806KHS\0806KHS-V4\montre\EDA.cs

,Numero_inclusion,Chemin,Feuille
0,0801HDR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0801HDR\0801HDR - V4\Montre\EDA.csv,SAINTE-MARGUERITE
1,0802LAS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0802LAS\0802LAS - V4\Montre\EDA.csv,SAINTE-MARGUERITE
2,0803DPS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0803DPS\0803DPS-V4\Montre\EDA.csv,SAINTE-MARGUERITE
3,0804GOR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0804GOR\0804GOR_V4\Montre\EDA.csv,SAINTE-MARGUERITE
4,0805BMS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0805BMS\0805BMS - V4\Montre\EDA.csv,SAINTE-MARGUERITE
5,0806KHS,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0806KHS\0806KHS-V4\montre\EDA.csv,SAINTE-MARGUERITE
6,0807OMR,C:\Users\judupont\Desktop\AGING_19_01_2026\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_StMarguerite\0807OMR\V4\montre\EDA.csv,SAINTE-MARGUERITE


## CGD

In [26]:
# ===== Racine CGD =====
racine_cgd = (
    r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie"
    r"\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD"
)

# ===== Candidats attendus =====
candidats_cgd = [
    c.upper() for c in resultats_globaux["CGD"]["valides"]
]

rows_cgd = []

# =========================================================
# BOUCLE PRINCIPALE
# =========================================================
for root, dirs, files in os.walk(racine_cgd):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== identification =====
        pid = None
        for c in candidats_cgd:
            if c in chemin_upper:
                pid = c
                break

        if not pid:
            continue

        rows_cgd.append({
            "Numero_inclusion": pid,
            "Chemin": chemin,
            "Feuille": "CGD"
        })

        print(f"OK [{pid}] :", chemin)

# =========================================================
# DATAFRAME + SUPPRESSION DES DOUBLONS
# =========================================================
df_cgd = pd.DataFrame(rows_cgd)

df_cgd = (
    df_cgd
    .sort_values("Chemin")
    .drop_duplicates(subset="Numero_inclusion", keep="first")
    .reset_index(drop=True)
)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus CGD : {len(candidats_cgd)}")
print(f"Fichiers EDA retenus   : {len(df_cgd)}")

manquants = sorted(
    set(candidats_cgd) - set(df_cgd["Numero_inclusion"])
)

if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CGD sont présents")

display(df_cgd)

OK [0901SMR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0901SMR\0901SMR - V4\Montre\1671008068_A01115\EDA.csv
OK [0902SIS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0902SIS\0902SIS-V4\montre\1675242525_A01115\EDA.csv
OK [0903DJS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0903DJS\V4\montre\1678265782_A01115\EDA.csv
OK [0904KJS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS-V4\montre\1679472202_A01115\EDA.csv
OK [0905SLR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V4\montre\1680680968_A01115\EDA.csv

Candidats attendus CGD : 5
Fichiers EDA retenus   : 5
✅ Tous les candidats CGD sont présents


,Numero_inclusion,Chemin,Feuille
0,0901SMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0901SMR\0901SMR - V4\Montre\1671008068_A01115\EDA.csv,CGD
1,0902SIS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0902SIS\0902SIS-V4\montre\1675242525_A01115\EDA.csv,CGD
2,0903DJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0903DJS\V4\montre\1678265782_A01115\EDA.csv,CGD
3,0904KJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0904KJS\0904KJS-V4\montre\1679472202_A01115\EDA.csv,CGD
4,0905SLR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_CGD\0905SLR\0905SLR-V4\montre\1680680968_A01115\EDA.csv,CGD


## CERCA

In [28]:
# ===== Racine CERCA =====
racine_cerca = r"C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA"

# ===== Candidats CERCA =====
candidats_cerca = [c.upper() for c in resultats_globaux['CERCA']['valides']]

# ===== Liste pour stocker les fichiers EDA =====
rows = []

# =========================================================
# BOUCLE PRINCIPALE 
# =========================================================
for root, dirs, files in os.walk(racine_cerca):

    if "V4" not in root.upper():
        continue

    for f in files:

        if not f.lower().endswith(".csv"):
            continue
        if "EDA" not in f.upper():
            continue

        chemin = os.path.join(root, f)
        chemin_upper = chemin.upper()

        # ===== Identification =====
        pid_trouve = None
        for c in candidats_cerca:
            if c in chemin_upper:
                pid_trouve = c
                break

        if not pid_trouve:
            print(f"❌ PID NON TROUVÉ : {chemin}")
            continue

        rows.append({
            "Numero_inclusion": pid_trouve,
            "Chemin": chemin,
            "Feuille": "CERCA"
        })

        print(f"✅ EDA trouvé [{pid_trouve}] :", chemin)

# =========================================================
# DATAFRAME
# =========================================================
df_cerca_final = pd.DataFrame(rows)
df_cerca_final = df_cerca_final.drop_duplicates(subset="Numero_inclusion", keep="first").reset_index(drop=True)

# =========================================================
# AFFICHAGE
# =========================================================
print("\n====================================")
print(f"Candidats attendus CERCA : {len(candidats_cerca)}")
print(f"Fichiers EDA retenus CERCA : {len(df_cerca_final)}")

manquants = sorted(set(candidats_cerca) - set(df_cerca_final["Numero_inclusion"]))
if manquants:
    print("⚠️ Candidats manquants :", manquants)
else:
    print("✅ Tous les candidats CERCA sont présents")

display(df_cerca_final)

✅ EDA trouvé [0401BDS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0401BDS\V4\Montre\1573634231_A0117F\EDA.csv
✅ EDA trouvé [0405RCR] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0405RCR\V4\Montre\EDA.csv
✅ EDA trouvé [0406BBS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0406BBS\V4\Montre\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0407HMS\V4\Montre\EDA.csv
✅ EDA trouvé [0408SJS] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0408SJS\V4\Montre\1574847506_A0117F\EDA.csv
✅ EDA trouvé [0404DER] : C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0404DER\V4\Montre\1573202858_A0117F\EDA.csv
❌ PID NON TROUVÉ : C:\Users\judupont\Desktop\AGIN

,Numero_inclusion,Chemin,Feuille
0,0401BDS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0401BDS\V4\Montre\1573634231_A0117F\EDA.csv,CERCA
1,0405RCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0405RCR\V4\Montre\EDA.csv,CERCA
2,0406BBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0406BBS\V4\Montre\EDA.csv,CERCA
3,0408SJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Alizé GRANGE\0408SJS\V4\Montre\1574847506_A0117F\EDA.csv,CERCA
4,0404DER,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0404DER\V4\Montre\1573202858_A0117F\EDA.csv,CERCA
5,0409HCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0409HCR\V4\Montre\EDA.csv,CERCA
6,0410FMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\GUILLOU\GUILLOU\CeRCA POITIERS\Camille GUILLOU\0410FMS\V4\Montre\EDA.csv,CERCA
7,0413MMR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\RIGALLEAU\RIGALLEAU\participant Sarah\0413MMR\V4\Montre\EDA.csv,CERCA
8,0414PVR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\RIGALLEAU\RIGALLEAU\participant Sarah\0414PVR\V4\Montre\EDA.csv,CERCA
9,0420MCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\CERCA\RIGALLEAU\RIGALLEAU\participant Sarah\0420MCR\V4\Montre\EDA.csv,CERCA


## CONCATENATION

In [29]:
# ===== Liste des DataFrames par feuille =====
dfs = [
    df_aphm_final,  
    df_caen_final,
    df_poitiers_final,     
    df_rouen_final,
    df_laveran_final,
    df_lpc_final,         
    df_lpc2_final,
    df_sm,          
    df_cgd,         
    df_cerca_final      
]

# ===== Normalisation des colonnes =====
for i, df in enumerate(dfs):
    if "PID" in df.columns:
        df.rename(columns={"PID": "Numero_inclusion"}, inplace=True)
    # Assurer que la colonne Condition existe
    if "Condition" not in df.columns:
        df["Condition"] = None
    # On garde uniquement les colonnes essentielles
    df = df[["Numero_inclusion", "Chemin", "Feuille"]]
    dfs[i] = df

# ===== Concatenation =====
df_global = pd.concat(dfs, ignore_index=True)

# ===== Suppression des doublons par Numero_inclusion =====
df_global = df_global.drop_duplicates(subset=["Numero_inclusion"], keep="first")

# ===== Tri par Numero_inclusion =====
df_global = df_global.sort_values("Numero_inclusion").reset_index(drop=True)

# ===== Résumé =====
print(f"Total candidats uniques : {df_global['Numero_inclusion'].nunique()}")
print(f"Total fichiers conservés : {len(df_global)}")
display(df_global)


Total candidats uniques : 132
Total fichiers conservés : 132


,Numero_inclusion,Chemin,Feuille
0,0101CAR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0101CAR\0101CAR 04-04-2019 V4\Montre 0101CAR\1554360113_A01093\EDA.csv,APHM
1,0101EMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0101EMS\0101EMS V4\Montre\1571730172_A01115\EDA.csv,LPC
2,0102EMS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0102EMS\V4 0102EMS\Montre\1567668999_A01115\EDA.csv,LPC
3,0102PCR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0102PCR\0102PCR 12-04-2019 V4\Montre 0102PCR\1555052819_A01093\EDA.csv,APHM
4,0103BPS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0103BPS\V4 0103BPS\Montre\1568706200_A01115\EDA.csv,LPC
5,0104FJS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0104FJS\0104FJS 08-11-2019 V4\Montre 0104FJS\1573198120_A01093\EDA.csv,APHM
6,0104IBS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0104IBS\V4 0104IBS\Montre\EDA.csv,LPC
7,0105PNR,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0105PNR\0105PNR 09-09-2019 V4\Montre 0105PNR\1568012045_A01093\EDA.csv,APHM
8,0107DSS,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LPC\GAUTHIER\GAUTHIER\PATIENTS\TOUS_patients_APHM_laveran\0107DSS\0107DSS 17-10-2019 V4\Montre 0107DSS\1571296181_A01093\EDA.csv,APHM
9,0107LER,C:\Users\judupont\Desktop\AGING_19_01_2026_copie\LNSC\DESHAYES\DESHAYES\DATA_Aging\0107LER\V4 0107LER\Montre\1567584597_A01115\EDA.csv,LPC


In [30]:
# CONSERVATION DES FEUILLES DANS UN FICHEIR TXT
chemin_txt = r"C:\Users\judupont\Desktop\df_global_v4_EDA.txt"

df_global.to_csv(
    chemin_txt,
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

print(f"📄 df_global sauvegardé : {chemin_txt}")

📄 df_global sauvegardé : C:\Users\judupont\Desktop\df_global_v4_EDA.txt


# Verif temps unix

In [ ]:
# Exemple de timestamp Unix
timestamp_unix = 1554360113

# Conversion en datetime
dt = datetime.datetime.fromtimestamp(timestamp_unix)

print(dt)

# Verification que le temps d'experience et le temps du fichier EDA correspondent, offset et continuité

In [31]:
def convertir_heure_en_secondes(x):
    if pd.isna(x):
        return None

    # Excel → heure uniquement
    if isinstance(x, (int, float)):
        seconds = float(x) * 24 * 3600
        return seconds % (24 * 3600)

    try:
        t = pd.to_datetime(str(x).strip()).time()
        return t.hour * 3600 + t.minute * 60 + t.second
    except Exception:
        return None



dfs_excel = []

for feuille, data in resultats_bloc1.items():

    df_excel = data["df"].copy()

    # filtrage candidats retenus
    df_excel["Numero_inclusion"] = df_excel["Numero_inclusion"].astype(str)
    df_excel = df_excel[
        df_excel["Numero_inclusion"].isin(df_global['Numero_inclusion'])
    ]

    if df_excel.empty:
        continue

    # Conversion heures → secondes
    df_excel["heure_debut_sec"] = df_excel["heure_montre_v4"].apply(convertir_heure_en_secondes)
    df_excel["heure_fin_sec"] = df_excel["heure_fin_tests_v4"].apply(convertir_heure_en_secondes)

    # Durée visite Excel
    df_excel["duree_experience_sec"] = (
        df_excel["heure_fin_sec"] - df_excel["heure_debut_sec"]
    )

    # Sécurité passage minuit
    df_excel.loc[
        df_excel["duree_experience_sec"] < 0,
        "duree_experience_sec"
    ] += 24 * 3600

    df_excel["Feuille"] = feuille
    dfs_excel.append(df_excel)

df_excel_global = pd.concat(dfs_excel, ignore_index=True)


print(f"Lignes Excel retenues : {len(df_excel_global)}")
display(df_excel_global[[
    "Numero_inclusion",
    "Feuille",
    "duree_experience_sec"
]])

Lignes Excel retenues : 132


,Numero_inclusion,Feuille,duree_experience_sec
0,0101CAR,APHM,9660
1,0102PCR,APHM,7440
2,0105PNR,APHM,7080
3,0104FJS,APHM,6120
4,0107DSS,APHM,6060
5,0110LPR,APHM,8700
6,0111MNR,APHM,7560
7,0112BSR,APHM,8220
8,0114LMS,APHM,6960
9,0116VAR,APHM,10020


# D'ou vient l'écart ? L'offest doit etre appliqué en debut, en fin, au milieu?

In [32]:
def sec_to_datetime(sec, date_ref):
    return datetime.combine(
        date_ref.date(),
        datetime.min.time()
    ) + timedelta(seconds=int(sec))


resultats_offset_eda = []

for _, row in df_global.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()
    chemin_eda = row["Chemin"]

    # ===== Lecture EDA =====
    try:
        if ".ZIP|" in chemin_eda.upper():
            zip_path, inner_csv = chemin_eda.split("|", 1)
            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_raw = pd.read_csv(f, header=None)
        else:
            df_raw = pd.read_csv(chemin_eda, header=None)

    except Exception as e:
        print(f"⚠️ Lecture EDA impossible pour {pid} : {e}")
        continue

    if len(df_raw) < 3:
        continue

    # ===== Reconstruction Timestamp =====
    try:
        t0_unix = float(df_raw.iloc[0, 0])
        freq = float(df_raw.iloc[1, 0])
    except:
        continue

    if freq <= 0:
        continue

    df_eda = df_raw.iloc[2:].copy()
    df_eda.columns = ["EDA"]
    df_eda["EDA"] = pd.to_numeric(df_eda["EDA"], errors="coerce")
    df_eda = df_eda.dropna()

    if df_eda.empty:
        continue

    pas = 1 / 4
    n = len(df_eda)

    t_debut = datetime.fromtimestamp(t0_unix)
    df_eda["Timestamp"] = [
        t_debut + timedelta(seconds=i * pas)
        for i in range(n)
    ]

    t_min = df_eda["Timestamp"].min()
    t_max = df_eda["Timestamp"].max()

    # ===== Excel =====
    ligne_excel = df_excel_global.loc[
        df_excel_global["Numero_inclusion"] == pid
    ]

    if ligne_excel.empty:
        continue

    h_debut_sec = ligne_excel["heure_debut_sec"].values[0]
    h_fin_sec   = ligne_excel["heure_fin_sec"].values[0]
    duree_excel_sec = ligne_excel["duree_experience_sec"].values[0]

    if pd.isna(h_debut_sec) or pd.isna(h_fin_sec) or pd.isna(duree_excel_sec):
        continue

    h_debut = sec_to_datetime(h_debut_sec, t_min)
    h_fin   = sec_to_datetime(h_fin_sec, t_min)

    # =====================================================
    # Calcul du retard si EDA commence après Excel
    # =====================================================
    if t_min > h_debut:
        delta_retard_sec = (t_min - h_debut).total_seconds()
    else:
        delta_retard_sec = 0

    # ===== Comptages =====
    nb_avant = (df_eda["Timestamp"] < h_debut).sum()
    nb_dans = (
        (df_eda["Timestamp"] >= h_debut) &
        (df_eda["Timestamp"] <= h_fin)
    ).sum()
    nb_apres = (df_eda["Timestamp"] > h_fin).sum()

    total = len(df_eda)

    if nb_apres > nb_avant:
        origine = "FIN"
    elif nb_avant > nb_apres:
        origine = "DEBUT"
    else:
        origine = "MIXTE"

    resultats_offset_eda.append({
        "Numero_inclusion": pid,
        "nb_total_points": total,
        "nb_avant_excel": nb_avant,
        "nb_dans_excel": nb_dans,
        "nb_apres_excel": nb_apres,
        "origine_offset": origine,
        "delta_retard_sec": delta_retard_sec
    })

df_offset_eda = pd.DataFrame(resultats_offset_eda)

print(f"Offsets EDA calculés : {len(df_offset_eda)}")
display(df_offset_eda)

Offsets EDA calculés : 132


,Numero_inclusion,nb_total_points,nb_avant_excel,nb_dans_excel,nb_apres_excel,origine_offset,delta_retard_sec
0,0101CAR,39066,0,36749,2317,FIN,473.0
1,0101EMS,32076,32,31441,603,FIN,0.0
2,0102EMS,19662,0,19662,0,MIXTE,39.0
3,0102PCR,30570,0,29525,1045,FIN,59.0
4,0103BPS,28356,0,28001,355,FIN,20.0
5,0104FJS,25314,0,24321,993,FIN,40.0
6,0104IBS,27918,8,27601,309,FIN,0.0
7,0105PNR,28668,0,28301,367,FIN,5.0
8,0107DSS,24696,0,24077,619,FIN,41.0
9,0107LER,27414,12,27121,281,FIN,0.0


In [34]:
# ===== Dossier de sortie =====
desktop = Path.home() / "Desktop"
output_dir = desktop / "EDA_v4_tronqués"
output_dir.mkdir(exist_ok=True)

print(f"📁 Dossier de sortie : {output_dir}")

# ===== Boucle troncature =====
for _, row in df_offset_eda.iterrows():

    pid = str(row["Numero_inclusion"]).strip().upper()

    # récupérer chemin EDA depuis df_global
    ligne_global = df_global[df_global["Numero_inclusion"] == pid]
    if ligne_global.empty:
        continue

    chemin_eda = ligne_global.iloc[0]["Chemin"]

    nb_avant = int(row["nb_avant_excel"])
    nb_apres = int(row["nb_apres_excel"])

    # ===== Lecture EDA (ZIP ou non) =====
    try:
        if ".ZIP|" in chemin_eda.upper():
            zip_path, inner_csv = chemin_eda.split("|", 1)

            with zipfile.ZipFile(zip_path, "r") as z:
                with z.open(inner_csv) as f:
                    df_eda = pd.read_csv(f)
        else:
            df_eda = pd.read_csv(chemin_eda)

    except Exception as e:
        print(f"❌ Lecture impossible {pid}: {e}")
        continue

    total = len(df_eda)

    # ===== Indices de coupe =====
    # On conserve les 2 premières lignes, troncage à partir de la 3ème
    start = 2 + nb_avant
    end = total - nb_apres

    if start >= end:
        print(f"⚠️ Troncature invalide pour {pid} (start={start}, end={end})")
        continue

    # On concatène les 2 premières lignes intactes + le reste tronqué
    df_eda_trunc = pd.concat([df_eda.iloc[:2], df_eda.iloc[start:end]]).reset_index(drop=True)

    if df_eda_trunc.empty:
        print(f"⚠️ {pid} → fichier vide après troncature")
        continue

    # ===== Nom fichier de sortie =====
    if ".ZIP|" in chemin_eda.upper():
        nom_base = Path(inner_csv).name
    else:
        nom_base = Path(chemin_eda).name

    nom_fichier = f"offset_{pid}_{nom_base}"
    path_sortie = output_dir / nom_fichier

    # ===== Sauvegarde =====
    df_eda_trunc.to_csv(path_sortie, index=False)

    print(
        f"✅ {pid} | "
        f"avant={nb_avant}, après={nb_apres} | "
        f"{len(df_eda_trunc)} lignes → {path_sortie.name}"
    )

print("🎯 Troncature terminée")

📁 Dossier de sortie : C:\Users\judupont\Desktop\EDA_v4_tronqués
✅ 0101CAR | avant=0, après=2317 | 36750 lignes → offset_0101CAR_EDA.csv
✅ 0101EMS | avant=32, après=603 | 31442 lignes → offset_0101EMS_EDA.csv
✅ 0102EMS | avant=0, après=0 | 19663 lignes → offset_0102EMS_EDA.csv
✅ 0102PCR | avant=0, après=1045 | 29526 lignes → offset_0102PCR_EDA.csv
✅ 0103BPS | avant=0, après=355 | 28002 lignes → offset_0103BPS_EDA.csv
✅ 0104FJS | avant=0, après=993 | 24322 lignes → offset_0104FJS_EDA.csv
✅ 0104IBS | avant=8, après=309 | 27602 lignes → offset_0104IBS_EDA.csv
✅ 0105PNR | avant=0, après=367 | 28302 lignes → offset_0105PNR_EDA.csv
✅ 0107DSS | avant=0, après=619 | 24078 lignes → offset_0107DSS_EDA.csv
✅ 0107LER | avant=12, après=281 | 27122 lignes → offset_0107LER_EDA.csv
✅ 0108MMS | avant=0, après=247 | 23418 lignes → offset_0108MMS_EDA.csv
✅ 0109MJR | avant=0, après=183 | 25498 lignes → offset_0109MJR_EDA.csv
✅ 0110LPR | avant=0, après=0 | 33349 lignes → offset_0110LPR_EDA.csv
✅ 0110RMS | a

In [35]:
# ===== Paramètres =====
min_lignes = 10000
candidats_a_exclure = {"0330RTR", "0626MCS"} # trop court par rapport au temps d'experience 

dossier = Path.home() / "Desktop" / "EDA_v4_tronqués"

print(f"📂 Dossier analysé : {dossier}")

supprimes = []
conserves = []

for fichier in dossier.glob("*.csv"):

    nom = fichier.name

    # ===== Extraction PID depuis le nom =====
    match = re.search(r"\d{4}[A-Z]{3}", nom.upper())
    if not match:
        print(f"⚠️ PID introuvable dans {nom}")
        continue

    pid = match.group()

    # ===== Exclusion candidat spécifique =====
    if pid in candidats_a_exclure:
        fichier.unlink()
        supprimes.append((nom, "candidat exclu"))
        print(f"🗑️ {nom} → candidat exclu ({pid})")
        continue

    # ===== Comptage rapide des lignes =====
    try:
        with open(fichier, "r", encoding="utf-8") as f:
            n_lignes = sum(1 for _ in f)
    except Exception as e:
        print(f"❌ Lecture impossible {nom}: {e}")
        continue

    # ===== Exclusion fichiers trop courts =====
    if n_lignes < min_lignes:
        fichier.unlink()
        supprimes.append((nom, f"{n_lignes} lignes"))
        print(f"❌ {nom} → trop petit ({n_lignes} lignes)")
    else:
        conserves.append((nom, n_lignes))
        print(f"✅ {nom} → conservé ({n_lignes} lignes)")

# ===== Résumé =====
print("\n====== RÉSUMÉ ======")
print(f"Fichiers supprimés : {len(supprimes)}")
print(f"Fichiers conservés : {len(conserves)}")

if supprimes:
    print("\nDétail suppressions :")
    for s in supprimes:
        print(" -", s)


📂 Dossier analysé : C:\Users\judupont\Desktop\EDA_v4_tronqués
✅ offset_0101CAR_EDA.csv → conservé (36751 lignes)
✅ offset_0101EMS_EDA.csv → conservé (31443 lignes)
✅ offset_0102EMS_EDA.csv → conservé (19664 lignes)
✅ offset_0102PCR_EDA.csv → conservé (29527 lignes)
✅ offset_0103BPS_EDA.csv → conservé (28003 lignes)
✅ offset_0104FJS_EDA.csv → conservé (24323 lignes)
✅ offset_0104IBS_EDA.csv → conservé (27603 lignes)
✅ offset_0105PNR_EDA.csv → conservé (28303 lignes)
✅ offset_0107DSS_EDA.csv → conservé (24079 lignes)
✅ offset_0107LER_EDA.csv → conservé (27123 lignes)
✅ offset_0108MMS_EDA.csv → conservé (23419 lignes)
✅ offset_0109MJR_EDA.csv → conservé (25499 lignes)
✅ offset_0110LPR_EDA.csv → conservé (33350 lignes)
✅ offset_0110RMS_EDA.csv → conservé (23115 lignes)
✅ offset_0111MNR_EDA.csv → conservé (29948 lignes)
✅ offset_0111TAS_EDA.csv → conservé (22779 lignes)
✅ offset_0112BSR_EDA.csv → conservé (32771 lignes)
✅ offset_0112DRR_EDA.csv → conservé (24591 lignes)
✅ offset_0113DGR_EDA

# On peut passer au découpage!

In [36]:
# Calcul de la durée de l'écriture pour shift le debut du bloc 2, 7 minutes pour les candidats ayant un Nr dans l'une des deux colonnes de calcul 

# ===== PARAMÈTRE =====
DUREE_MOYENNE_ECRITURE_SEC = 7 * 60  # 7 minutes

col_debut = "heure_anamnese_fin_v4"
col_fin   = "heure_rlri16imm_debut_v4"

df = df_excel_global.copy()

# ===== Détection valeur non horaire =====
def est_non_horaire(val):
    if pd.isna(val):
        return True
    if isinstance(val, str):
        v = val.strip().upper()
        return v in {"NR", "N/R", "NON RENSEIGNE", ""}
    return False

# ===== Conversion vers Timestamp =====
def convertir_heure(val):
    if est_non_horaire(val):
        return pd.NaT

    # Excel numérique (fraction de jour)
    if isinstance(val, (int, float)):
        seconds = (float(val) * 24 * 3600) % (24 * 3600)
        return pd.Timestamp("1900-01-01") + pd.to_timedelta(seconds, unit="s")

    # datetime / Timestamp
    if isinstance(val, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=val.hour, minute=val.minute, second=val.second
        )

    try:
        t = pd.to_datetime(str(val).strip(), errors="coerce")
        if pd.isna(t):
            return pd.NaT
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=t.hour, minute=t.minute, second=t.second
        )
    except Exception:
        return pd.NaT

# ===== Conversion des colonnes =====
df["_ecriture_debut_ts"] = df[col_debut].apply(convertir_heure)
df["_ecriture_fin_ts"]   = df[col_fin].apply(convertir_heure)

# ===== Calcul durée =====
def calcul_duree(row):
    debut = row["_ecriture_debut_ts"]
    fin   = row["_ecriture_fin_ts"]

    if pd.notna(debut) and pd.notna(fin):
        delta = (fin - debut).total_seconds()

        # sécurité : durée négative ou absurde
        if 0 <= delta < 3600:
            return int(delta)

    return DUREE_MOYENNE_ECRITURE_SEC

df["duree_ecriture_sec"] = df.apply(calcul_duree, axis=1)

# ===== Diagnostic =====
nb_moyenne = (df["duree_ecriture_sec"] == DUREE_MOYENNE_ECRITURE_SEC).sum()

print("===== DURÉE ÉCRITURE CALCULÉE =====")
print(f"Total candidats : {len(df)}")
print("Le seul caniddat avec une assigantion de temps par défaut est le 0410FMS")

# ===== Nettoyage colonnes temporaires =====
df = df.drop(columns=["_ecriture_debut_ts", "_ecriture_fin_ts"])

# ===== Mise à jour dataframe principal =====
df_excel_global["duree_ecriture_sec"] = df["duree_ecriture_sec"]

display(
    df_excel_global[[
        "Numero_inclusion",
        col_debut,
        col_fin,
        "duree_ecriture_sec"
    ]]
)

===== DURÉE ÉCRITURE CALCULÉE =====
Total candidats : 132
Durées réelles calculées : 121
Durées remplacées par la moyenne (7 min) : 11


,Numero_inclusion,heure_anamnese_fin_v4,heure_rlri16imm_debut_v4,duree_ecriture_sec
0,0101CAR,2026-03-10 08:54:00,09:07:00,780
1,0102PCR,2026-03-10 09:20:00,09:30:00,600
2,0105PNR,2026-03-10 09:08:00,09:22:00,840
3,0104FJS,2026-03-10 08:44:00,08:50:00,360
4,0107DSS,2026-03-10 09:33:00,09:45:00,720
5,0110LPR,2026-03-10 09:19:00,09:36:00,1020
6,0111MNR,2026-03-10 09:37:00,09:47:00,600
7,0112BSR,2026-03-10 10:10:00,10:25:00,900
8,0114LMS,2026-03-10 10:12:00,10:21:00,540
9,0116VAR,2026-03-10 10:00:00,10:18:00,1080


In [38]:
def decouper_eda_par_bloc(df_eda, duree_bloc1, duree_bloc2, duree_bloc3, delta, duree_ecriture):
    """
    Découpe un EDA en 3 blocs en prenant en compte le retard (delta)
    pour ajuster la durée du bloc 1.
    """

    df_eda = df_eda.copy()
    
    if len(df_eda) < 3:
        print(f"⚠️ {pid} : Fichier trop court pour découpages")
        return None

    # ===== Lire timestamp initial et fréquence =====
    ts_unix = df_eda.iloc[0, 0]  # première ligne, première colonne
    freq_hz = df_eda.iloc[1, 0]  # deuxième ligne, première colonne

    try:
        t0 = datetime.datetime.fromtimestamp(float(ts_unix))
        delta_sec = 1.0 / 4  # fréquence EDA fixe 4 Hz
    except Exception as e:
        print(f"⚠️ {pid} : Impossible de lire timestamp/freq : {e}")
        return None

    # ===== Générer colonne Timestamp =====
    nb_lignes = len(df_eda) - 2
    timestamps = [t0 + datetime.timedelta(seconds=i * delta_sec) for i in range(nb_lignes)]
    df_eda_data = df_eda.iloc[2:].copy()
    df_eda_data["Timestamp"] = timestamps
    t_fin_rr = df_eda_data["Timestamp"].max()

    # ===== Récupérer durées des blocs =====
    durees = get_durees_blocs(pid, feuille)
    duree_bloc1 = durees["bloc1_sec"]
    duree_ecriture = durees["duree_ecriture_sec"]
    duree_bloc2 = durees["bloc2_sec"]
    duree_bloc3 = durees["bloc3_sec"]

    # ===== Récupérer delta pour ce PID =====
    delta_row = df_offset_eda.loc[df_offset_eda["Numero_inclusion"].str.upper() == pid.upper()]
    if delta_row.empty:
        delta = 0
    else:
        delta = float(delta_row["delta_retard_sec"].values[0])

    # ===== Bornes temporelles avec delta =====
    fin_bloc1 = t0 + pd.Timedelta(seconds=max(0, duree_bloc1 - delta))
    debut_bloc2 = fin_bloc1 + pd.Timedelta(seconds=duree_ecriture)
    fin_bloc2 = debut_bloc2 + pd.Timedelta(seconds=duree_bloc2)
    fin_bloc3 = fin_bloc2 + pd.Timedelta(seconds=duree_bloc3)
    fin_bloc3 = min(fin_bloc3, t_fin_rr)  # sécurité

    # ===== Découpage =====
    blocs = {
        "bloc1": df_eda_data[(df_eda_data["Timestamp"] >= t0) & (df_eda_data["Timestamp"] < fin_bloc1)],
        "bloc2": df_eda_data[(df_eda_data["Timestamp"] >= debut_bloc2) & (df_eda_data["Timestamp"] < fin_bloc2)],
        "bloc3": df_eda_data[(df_eda_data["Timestamp"] >= fin_bloc2) & (df_eda_data["Timestamp"] <= fin_bloc3)],
        "bornes": {
            "t0": t0,
            "fin_bloc1": fin_bloc1,
            "debut_bloc2": debut_bloc2,
            "fin_bloc2": fin_bloc2,
            "fin_bloc3": fin_bloc3,
            "t_fin_rr": t_fin_rr,
            "delta_sec": delta
        }
    }

    return blocs

In [39]:
def convertir_et_normaliser_heure(x):
    """
    Convertit n'importe quel format d'heure Excel / texte / datetime en pd.Timestamp
    normalisé sur le 1900-01-01, ne gardant que l'heure, minute, seconde.
    """
    if pd.isna(x):
        return pd.NaT

    # Excel numérique → fraction de jour ou date complète
    if isinstance(x, (int, float)):
        seconds = (float(x) * 24 * 3600) % (24*3600)
        return pd.Timestamp("1900-01-01") + pd.to_timedelta(seconds, unit="s")

    # Timestamp ou datetime → on garde juste l'heure
    if isinstance(x, (pd.Timestamp, datetime.datetime)):
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=x.hour, minute=x.minute, second=x.second
        )

    # Texte
    try:
        t = pd.to_datetime(str(x).strip(), errors="coerce")
        if pd.isna(t):
            return pd.NaT
        return pd.Timestamp(
            year=1900, month=1, day=1,
            hour=t.hour, minute=t.minute, second=t.second
        )
    except Exception:
        return pd.NaT


In [40]:
def get_durees_blocs(pid, feuille):
    """
    Récupère les durées pour les 3 blocs et la durée d'écriture à partir de df_excel_global
    pour un PID donné et une feuille donnée.
    
    La durée d'écriture sert à décaler le début du bloc 2.
    Si la durée n'est pas renseignée ou NR, on met 7 minutes (420 s) par défaut.
    """
    pid = str(pid).upper()
    
    # ===== Bloc 1 =====
    df_b1 = pd.DataFrame(resultats_bloc1[feuille]["df"])
    ligne = df_b1.loc[df_b1["Numero_inclusion"].str.upper() == pid]
    if ligne.empty:
        raise ValueError(f"PID {pid} absent du bloc 1 dans la feuille {feuille}")
    d1 = ligne["duree_bloc1"].values[0]

    # ===== Durée écriture =====
    # On part de df_excel_global
    ligne_excel = df_excel_global.loc[df_excel_global["Numero_inclusion"].str.upper() == pid]
    if ligne_excel.empty:
        val_ecriture = 420  # défaut si absent
    else:
        val_ecriture = ligne_excel["duree_ecriture_sec"].values[0]
        if pd.isna(val_ecriture) or val_ecriture < 0:
            val_ecriture = 420  # défaut pour NR ou vide

    # ===== Bloc 2 =====
    df_b2 = pd.DataFrame(resultats_bloc2[feuille]["df"])
    d2 = df_b2.loc[df_b2["Numero_inclusion"].str.upper() == pid, "duree_bloc2"].values[0]

    # ===== Bloc 3 =====
    df_b3 = pd.DataFrame(resultats_bloc3[feuille]["df"])
    d3 = df_b3.loc[df_b3["Numero_inclusion"].str.upper() == pid, "duree_bloc3"].values[0]

    return {
        "bloc1_sec": d1 * 60,
        "duree_ecriture_sec": val_ecriture,  # durée spécifique du candidat
        "bloc2_sec": d2 * 60,
        "bloc3_sec": d3 * 60,
    }


In [41]:
# ===== Dossiers =====
input_dir = os.path.join(os.path.expanduser("~"), "Desktop", "EDA_v4_tronqués")
output_dir = os.path.join(os.path.expanduser("~"), "Desktop", "EDA_v4_tronque_blocs")
os.makedirs(output_dir, exist_ok=True)

resultats_decoupage = {}

# ===== Boucle UNIQUEMENT sur les fichiers tronqués =====
for fichier in os.listdir(input_dir):

    if not fichier.lower().endswith(".csv"):
        continue

    chemin_eda = os.path.join(input_dir, fichier)

    # ===== Extraction PID depuis le nom =====
    match = re.search(r"\d{4}[A-Z]{3}", fichier.upper())
    if not match:
        print(f"⚠️ PID introuvable dans {fichier}")
        continue

    pid = match.group()

    # ===== Récupération de la feuille associée =====
    ligne = df_global.loc[df_global["Numero_inclusion"] == pid]
    if ligne.empty:
        print(f"⚠️ Feuille introuvable pour {pid}")
        continue

    feuille = ligne["Feuille"].values[0]

    print(f"\n--- Découpage {pid} ({feuille}) ---")

    # ===== Lecture EDA =====
    try:
        df_rr = pd.read_csv(chemin_eda, header = None)
    except Exception as e:
        print(f"❌ Lecture impossible {pid} : {e}")
        continue

    # ===== Récupération des durées =====
    try:
        durees = get_durees_blocs(pid, feuille)
    except Exception as e:
        print(f"⚠️ Impossible de récupérer les durées pour {pid} ({feuille}) : {e}")
        continue
        
    delta_row = df_offset_eda.loc[df_offset_eda["Numero_inclusion"].str.upper() == pid.upper()]
    if delta_row.empty:
        delta_val = 0
    else:
        delta_val = float(delta_row["delta_retard_sec"].values[0])
        
    # ===== Découpage =====
    decoupe = decouper_eda_par_bloc(
        df_eda=df_rr,
        duree_ecriture=durees["duree_ecriture_sec"],
        duree_bloc1=durees["bloc1_sec"],
        duree_bloc2=durees["bloc2_sec"],
        duree_bloc3=durees["bloc3_sec"],
        delta = delta_val
    )

    if decoupe is None:
        print(f"⚠️ Découpage vide pour {pid}")
        continue

    # ===== Sauvegarde par bloc =====
    for bloc in ["bloc1", "bloc2", "bloc3"]:
        df_bloc = decoupe[bloc]

        if df_bloc.empty:
            print(f"⚠️ {pid} {bloc} vide")
            continue

        nom_sortie = f"{pid}_{bloc}.csv"
        df_bloc.to_csv(
            os.path.join(output_dir, nom_sortie),
            index=False
        )

    # ===== Stockage mémoire =====
    resultats_decoupage[pid] = decoupe

    print(
        f"✔ {pid} | "
        f"B1={len(decoupe['bloc1'])} | "
        f"B2={len(decoupe['bloc2'])} | "
        f"B3={len(decoupe['bloc3'])}"
    )

print("\n✅ Découpage terminé")
print("📁 Résultats dans :", output_dir)


--- Découpage 0101CAR (APHM) ---
✔ 0101CAR | B1=2908 | B2=17040 | B3=13680

--- Découpage 0101EMS (LPC) ---
✔ 0101EMS | B1=8640 | B2=9120 | B3=10800

--- Découpage 0102EMS (LPC) ---
✔ 0102EMS | B1=2244 | B2=6000 | B3=9017

--- Découpage 0102PCR (APHM) ---
✔ 0102PCR | B1=3124 | B2=14160 | B3=9840

--- Découpage 0103BPS (LPC) ---
✔ 0103BPS | B1=7120 | B2=8400 | B3=9360

--- Découpage 0104FJS (APHM) ---
✔ 0104FJS | B1=3680 | B2=9600 | B3=9600

--- Découpage 0104IBS (LPC) ---
✔ 0104IBS | B1=6960 | B2=7680 | B3=9600

--- Découpage 0105PNR (APHM) ---
✔ 0105PNR | B1=3340 | B2=10320 | B3=11280

--- Découpage 0107DSS (APHM) ---
✔ 0107DSS | B1=5596 | B2=9120 | B3=6480

--- Découpage 0107LER (LPC) ---
✔ 0107LER | B1=4800 | B2=9600 | B3=10320

--- Découpage 0108MMS (LPC) ---
✔ 0108MMS | B1=2296 | B2=7680 | B3=10560

--- Découpage 0109MJR (LPC) ---
✔ 0109MJR | B1=3416 | B2=8880 | B3=10560

--- Découpage 0110LPR (APHM) ---
✔ 0110LPR | B1=3348 | B2=12240 | B3=13679

--- Découpage 0110RMS (LPC) ---
✔

In [ ]:
## Traitment pour passer le fichier dans kubios